# Example 0.13.1 / 3: Exercise 4-4: Bakery Crews over 13 Periods

Companion notebook for **0.13.1 Exercises on Building Models** in the AMPL lab Documentation, section "Exercise 4-4: Bakery Crews over 13 Periods". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 4-4: Bakery Crews over 13 Periods

A bakery decides how many crews to employ on each shift in each of 13 four-week periods. Production
of each product $p$ in period $t$, in thousands of boxes, must cover demand $d_{pt}$; it takes
$r_p$ crew-hours per thousand boxes; each crew works $h_t$ hours in period $t$; at most $l = 8$
crews fit on a shift; a crew on shift $s$ costs $w_s$ per period; and total crews may change by
at most $M = 3$ between periods:

$$
\min \sum_{s,t} w_s Y_{st}
\quad\text{s.t.}\quad
\sum_p r_p X_{pt} \le h_t \sum_s Y_{st},\quad
-M \le \sum_s (Y_{s,t+1} - Y_{st}) \le M,\quad
X_{pt} \ge d_{pt},\quad 0 \le Y_{st} \le l.
$$

The change limit is a range constraint (Range and Logical Constraints (AMPL lab documentation, section 1.4.2)) over the
ordered periods `1..T-1`, as on Multiperiod Models (AMPL lab documentation, section 4.4.2).

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set SHIFTS;                           # day and night shifts
set PROD;                             # products
param T integer > 1;                  # number of four-week periods
param lines integer > 0;              # production lines: most crews per shift
param max_change >= 0;                # largest change in total crews between periods
param rate {PROD} > 0;                # crew-hours per 1000 boxes
param hours {1..T} > 0;               # hours a crew works in each period
param wage {SHIFTS} > 0;              # dollars per crew per period
param demand {PROD, 1..T} >= 0;       # 1000s of boxes required

var Make {p in PROD, t in 1..T} >= demand[p,t];  # 1000s of boxes baked
var Crews {SHIFTS, 1..T} >= 0, <= lines;         # crews employed

minimize Total_Wages: sum {s in SHIFTS, t in 1..T} wage[s] * Crews[s,t];  # dollars

# crew-hours needed fit within crew-hours employed
subject to Time {t in 1..T}:
    sum {p in PROD} rate[p] * Make[p,t] <= hours[t] * sum {s in SHIFTS} Crews[s,t];
# total crews change by at most max_change from one period to the next
subject to Change {t in 1..T-1}:
    -max_change <= sum {s in SHIFTS} (Crews[s,t+1] - Crews[s,t]) <= max_change;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

ampl.set["SHIFTS"] = ["day", "night"]
ampl.set["PROD"] = ["18REG", "24REG", "24PRO"]
ampl.param["T"] = 13
ampl.param["lines"] = 8
ampl.param["max_change"] = 3
ampl.param["rate"] = {"18REG": 1.194, "24REG": 1.509, "24PRO": 1.509}
ampl.param["wage"] = {"day": 44900, "night": 123100}
periods = pd.DataFrame(
    {"18REG": [63.8, 76.0, 88.4, 913.8, 115.0, 133.8, 79.6, 111.0, 121.6, 470.0, 78.4, 99.4, 140.4],
     "24REG": [1212.0, 306.2, 319.0, 208.4, 298.0, 328.2, 959.6, 257.6, 335.6, 118.0, 284.8, 970.0, 343.8],
     "24PRO": [0, 0, 0, 0, 0, 0, 0, 0, 0, 1102.0, 0, 0, 0],
     "hours": [156, 152, 160, 152, 156, 152, 152, 160, 152, 160, 160, 144, 144]},
    index=range(1, 14),
)
ampl.param["hours"] = periods["hours"]
ampl.param["demand"] = periods[["18REG", "24REG", "24PRO"]].T

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, round(ampl.obj["Total_Wages"].value(), 2))
crews = ampl.var["Crews"].to_dict()
for t in range(1, 14):
    print(t, round(crews["day", t], 3), round(crews["night", t], 3))

HiGHS 1.14.0: optimal solution; objective 7903918.605
6 simplex iterations
0 barrier iterations
solved 7903918.61
1 8 4.212
2 8 1.212
3 6.247 0
4 8 1.247
5 6.247 0
6 7.152 0
7 8 2.152
8 8 1.013
9 8 4.013
10 8 7.013
11 8 4.013
12 8 2.989
13 7.989 0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set SHIFTS := day night;
set PROD := 18REG 24REG 24PRO;
param T := 13;
param lines := 8;
param max_change := 3;
param rate := 18REG 1.194  24REG 1.509  24PRO 1.509;
param wage := day 44900  night 123100;
param demand (tr):  18REG   24REG   24PRO :=
1      63.8  1212.0     0
2      76.0   306.2     0
3      88.4   319.0     0
4     913.8   208.4     0
5     115.0   298.0     0
6     133.8   328.2     0
7      79.6   959.6     0
8     111.0   257.6     0
9     121.6   335.6     0
10    470.0   118.0  1102.0
11     78.4   284.8     0
12     99.4   970.0     0
13    140.4   343.8     0;
param hours := 1 156  2 152  3 160  4 152  5 156  6 152  7 152
               8 160  9 152  10 160  11 160  12 144  13 144;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Crews;

HiGHS 1.14.0: optimal solution; objective 7903918.605
6 simplex iterations
0 barrier iterations
Crews [*,*] (tr)
:      day      night     :=
1    8         4.21208
2    8         1.21208
3    6.24706   0
4    8         1.24706
5    6.24706   0
6    7.15183   0
7    8         2.15183
8    8         1.0135
9    8         4.0135
10   8         7.0135
11   8         4.0135
12   8         2.98898
13   7.98898   0
;

